In [1]:
from pathlib import Path
import networkx as nx
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from collections import Counter
import pipeline
import random
from scipy.stats import spearmanr
import igraph as ig

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100
orig = 0

In [2]:
import pickle

pkl_path  = Path('data/agg/agg_network_24h_undir.pkl')
csv_path  = Path('data/agg/agg_network_24h_undir.csv')
orig_path = Path('data/undir_trials/24h/0_network_24h.txt')

if orig:
    print('Using: Original Network')
    G = pipeline.load(orig_path)
elif pkl_path.exists():
    print(f'Using: Aggregated Network (pickle)  ← {pkl_path}')
    with open(pkl_path, 'rb') as f:
        G = pickle.load(f)
else:
    print(f'Using: Aggregated Network (CSV)  ← {csv_path}')
    df = pd.read_csv(csv_path, header=None)
    G = nx.from_pandas_edgelist(df, source=0, target=1, edge_attr=True)

Using: Aggregated Network (pickle)  ← data\agg\agg_network_24h_undir.pkl


# Descriptive Analysis

Goal: characterise the raw network structure to inform link prediction strategy — e.g. whether the graph is scale-free, how densely clustered it is, whether spatial proximity drives edges, and which POI types dominate.

## 1. Basic Topology

In [3]:
def characterize(G):
    n = G.number_of_nodes()
    m = G.number_of_edges()
    density = nx.density(G)

    degrees = np.array([d for _, d in G.degree()])
    avg_deg    = degrees.mean()
    max_deg    = int(degrees.max())
    median_deg = int(np.median(degrees))

    ccs = sorted(nx.connected_components(G), key=len, reverse=True)
    n_ccs    = len(ccs)
    lcc_size = len(ccs[0])
    lcc_frac = lcc_size / n
    avg_clust    = nx.average_clustering(G)
    # transitivity = nx.transitivity(G)

    # approximate diameter via double-sweep BFS
    def approx_diameter(H):
        src = next(iter(H.nodes()))
        lengths1 = nx.single_source_shortest_path_length(H, src)
        far1 = max(lengths1, key=lengths1.get)
        lengths2 = nx.single_source_shortest_path_length(H, far1)
        far2 = max(lengths2, key=lengths2.get)
        return lengths2[far2]

    # approximate aspl
    def estimate_aspl(G, sample_size=100):
        nodes = list(G.nodes())
        if len(nodes) <= sample_size:
            return nx.average_shortest_path_length(G)
        
        # Pick random source nodes
        sources = random.sample(nodes, sample_size)
        lengths = []
        
        for source in sources:
            # Get distances from source to all other nodes
            path_dict = nx.single_source_shortest_path_length(G, source)
            lengths.extend(path_dict.values())
            
        return np.mean(lengths)

    lcc_sub     = G.subgraph(ccs[0])
    # diam_approx = approx_diameter(lcc_sub)
    # aspl = estimate_aspl(lcc_sub)

    # modularity
    partition = nx.community.louvain_communities(lcc_sub)
    modularity = nx.community.modularity(lcc_sub, partition)

    print('=' * 50)
    print('  BASIC GRAPH STATISTICS')
    print('=' * 50)
    print(f'  Nodes                 : {n:>12,}')
    print(f'  Edges                 : {m:>12,}')
    print(f'  Density               : {density:>12.6f}')
    print(f'  Avg degree            : {avg_deg:>12.2f}')
    print(f'  Max degree            : {max_deg:>12,}')
    print(f'  Median degree         : {median_deg:>12,}')
    print(f'  Connected components  : {n_ccs:>12,}')
    print(f'  LCC (nodes)           : {lcc_size:>12,}  ({lcc_frac:.1%})')
    print(f'  LCC density           : {nx.density(lcc_sub):>12.6f}')
    # print(f'  Approx diameter (LCC) : {diam_approx:>12,}')
    # print(f'  Approx ASPL           : {aspl:>12.4f}')
    print(f'  Avg clustering coef   : {avg_clust:>12.4f}')
    print(f'  Highest modularity    : {modularity:>12.4f}')
    # print(f'  Transitivity          : {transitivity:>12.4f}')
    print('=' * 50)

print('ORIGINAL:\n')
characterize(G)

# --- null model comparison ---

# print('NULL:\n')
# deg_seq = [d for n, d in G.degree()]
# null_model_multi = nx.configuration_model(deg_seq)
# # Convert to simple graph to allow clustering metrics
# null_model = nx.Graph(null_model_multi) 
# null_model.remove_edges_from(nx.selfloop_edges(null_model)) # Optional: clean up loops
# characterize(null_model)

ORIGINAL:

  BASIC GRAPH STATISTICS
  Nodes                 :       15,025
  Edges                 :    5,239,856
  Density               :     0.046425
  Avg degree            :       697.48
  Max degree            :        8,600
  Median degree         :          460
  Connected components  :            1
  LCC (nodes)           :       15,025  (100.0%)
  LCC density           :     0.046425
  Avg clustering coef   :       0.4251
  Highest modularity    :       0.5586


## 2. Degree Distribution

In [3]:
degrees = np.array([d for _, d in G.degree()])

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# linear histogram
ax = axes[0]
ax.hist(degrees, bins=80, color='steelblue', edgecolor='white', linewidth=0.4)
ax.set_xlim(1)
ax.set_xlabel('Degree')
ax.set_ylabel('Node count')
ax.set_title('Degree Distribution')
ax.yaxis.set_major_formatter(
    ticker.FuncFormatter(lambda x, _: f'{x/1e3:.0f}k' if x >= 1000 else str(int(x)))
)

# log-log CCDF — a straight line here indicates power-law behaviour
ax = axes[1]
sorted_deg = np.sort(degrees)
ccdf = 1.0 - np.arange(1, len(sorted_deg) + 1) / len(sorted_deg)
ax.loglog(sorted_deg, ccdf, '.', markersize=2, color='steelblue', alpha=0.6)
ax.set_xlabel('Degree  (log scale)')
ax.set_ylabel('P(Degree ≥ k)  (log scale)')
ax.set_title('Degree CCDF (log-log)')

plt.suptitle('Degree Distribution', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'degree gini: {1 - 2*np.trapz(np.cumsum(np.sort(degrees))/degrees.sum(), np.linspace(0,1,len(degrees))):.3f}')

: 

### 2.1 deg-weight correlation

## 3. Node Attributes

In [ ]:
poi_types   = [data.get('poi_type', 'Unknown') for _, data in G.nodes(data=True)]
type_counts = Counter(poi_types)
type_df     = pd.DataFrame(type_counts.most_common(), columns=['POI Type', 'Count'])
type_df['Fraction'] = type_df['Count'] / type_df['Count'].sum()

palette = sns.color_palette('muted', len(type_df))

fig, ax = plt.subplots(figsize=(10, max(4, len(type_df) * 0.4)))
bars = ax.barh(type_df['POI Type'][::-1], type_df['Count'][::-1],
               color=palette[::-1], edgecolor='white', linewidth=0.4)
ax.set_xlabel('Number of nodes')
ax.set_title('POI Type Distribution', fontweight='bold')
ax.xaxis.set_major_formatter(
    ticker.FuncFormatter(lambda x, _: f'{x/1e3:.0f}k' if x >= 1000 else str(int(x)))
)
for bar, frac in zip(bars, type_df['Fraction'][::-1]):
    ax.text(bar.get_width() + type_df['Count'].max() * 0.005,
            bar.get_y() + bar.get_height() / 2,
            f'{frac:.1%}', va='center', fontsize=8)
plt.tight_layout()
plt.show()

print(type_df.to_string(index=False))

In [ ]:
unique_visits = np.array([d.get('unique_visits', 0) for _, d in G.nodes(data=True)], dtype=float)
total_visits  = np.array([d.get('total_visits',  0) for _, d in G.nodes(data=True)], dtype=float)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, raw, label, color in [
    (axes[0], unique_visits, 'Unique Visitors', 'teal'),
    (axes[1], total_visits,  'Total Visits',    'coral'),
]:
    log_data = np.log1p(raw[raw > 0])
    ax.hist(log_data, bins=60, color=color, edgecolor='white', linewidth=0.3, alpha=0.85)
    ax.set_xlabel(f'log(1 + {label})')
    ax.set_ylabel('Node count')
    ax.set_title(f'{label} (log-scaled)', fontweight='bold')
    med_raw = np.median(raw[raw > 0])
    ax.axvline(np.log1p(med_raw), color='black', linestyle='--', linewidth=1.2,
               label=f'median = {int(med_raw):,}')
    ax.legend(fontsize=9)

plt.suptitle('Node Visit Count Distributions', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Edge Attributes

In [ ]:
dep_vals      = np.array(list(nx.get_edge_attributes(G, 'DEP').values()),        dtype=float)
dist_vals     = np.array(list(nx.get_edge_attributes(G, 'DIST_KM').values()),    dtype=float)
covisit_vals  = np.array(list(nx.get_edge_attributes(G, 'N_COVISITS').values()), dtype=float)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# DEP
ax = axes[0]
dep_pos = dep_vals[dep_vals > 0]
ax.hist(np.log10(dep_pos), bins=60, color='mediumpurple', edgecolor='white', linewidth=0.3)
ax.set_xlabel('log₁₀(DEP)')
ax.set_xlim(min(np.log10(dep_pos)))
ax.set_ylabel('Edge count')
ax.set_title('Dependency Score', fontweight='bold')
ax.axvline(np.log10(np.median(dep_pos)), color='black', linestyle='--', linewidth=1.2,
           label=f'median = {np.median(dep_pos):.2e}')
ax.legend(fontsize=9)

# Co-visits
ax = axes[1]
ax.hist(np.log1p(covisit_vals), bins=60, color='seagreen', edgecolor='white', linewidth=0.3)
ax.set_xlabel('log(1 + Co-visits)')
ax.set_xlim(min(np.log1p(covisit_vals)))
ax.set_ylabel('Edge count')
ax.set_title('Co-visit Count', fontweight='bold')
ax.axvline(np.log1p(np.median(covisit_vals)), color='black', linestyle='--', linewidth=1.2,
           label=f'median = {int(np.median(covisit_vals))}')
ax.legend(fontsize=9)

plt.suptitle('Weight Distributions (log-scaled)', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print(f'DEP     — zero-weight edges: {(dep_vals == 0).sum():,}  ({(dep_vals == 0).mean():.1%})')
print(f'DIST    — range: [{dist_vals.min():.2f}, {dist_vals.max():.1f}] km,  median: {np.median(dist_vals):.2f} km')
print(f'COVISIT — range: [{int(covisit_vals.min())}, {int(covisit_vals.max())}],  median: {int(np.median(covisit_vals))}')
print('='*50)
degrees_dict = G.degree() 
edge_degrees = np.array([
    (degrees_dict[u] * degrees_dict[v])**0.5 
    for u, v in G.edges()
])
corr, p_val = spearmanr(edge_degrees, dep_vals)
print(f"Correlation (DEP): {corr:.4f} (p={p_val:.2e})")
corr, p_val = spearmanr(edge_degrees, covisit_vals)
print(f"Correlation (COV): {corr:.4f} (p={p_val:.2e})")

In [ ]:

# Sum of covisits per node vs degree
node_degrees = dict(G.degree())
sum_covisits = {n: 0.0 for n in G.nodes()}
for u, v, w in G.edges(data='N_COVISITS', default=0):
    sum_covisits[u] += w
    sum_covisits[v] += w

deg_arr = np.array([node_degrees[n] for n in G.nodes()])
cov_arr = np.array([sum_covisits[n] for n in G.nodes()])

# binned medians
deg_bins = np.logspace(np.log10(deg_arr.min()), np.log10(deg_arr.max()), 30)
bin_idx  = np.digitize(deg_arr, deg_bins)
bin_med_deg = []
bin_med_cov = []
for i in range(1, len(deg_bins)):
    mask = bin_idx == i
    if mask.sum() >= 5:
        bin_med_deg.append(np.median(deg_arr[mask]))
        bin_med_cov.append(np.median(cov_arr[mask]))

# power-law fit on log-log
log_d = np.log10(deg_arr[deg_arr > 0])
log_c = np.log10(cov_arr[deg_arr > 0] + 1)
slope, intercept = np.polyfit(log_d, log_c, 1)

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(deg_arr, cov_arr, s=3, alpha=0.15, color='steelblue', label='nodes')
ax.plot(bin_med_deg, bin_med_cov, 'o-', color='tomato', ms=5, lw=1.5, label='binned median')
fit_x = np.logspace(np.log10(deg_arr.min()), np.log10(deg_arr.max()), 200)
ax.plot(fit_x, 10**(intercept) * fit_x**slope, '--', color='black', lw=1.2,
        label=f'power-law fit  slope={slope:.2f}')
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('Degree (log scale)')
ax.set_ylabel('Sum of co-visits (log scale)')
ax.set_title('Node co-visit strength vs degree', fontweight='bold')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

rho, p = spearmanr(deg_arr, cov_arr)
print(f'Spearman ρ(degree, sum_covisits) = {rho:.4f}  (p={p:.2e})')
print(f'Power-law fit: sum_covisits ~ degree^{slope:.3f}')


## 7. Assortativity & Mixing

In [ ]:
deg_assort = nx.degree_assortativity_coefficient(G)
poi_assort = nx.attribute_assortativity_coefficient(G, 'poi_type')

print(f'Degree assortativity  : {deg_assort:+.4f}')
print(f'  (negative = hubs connect to low-degree nodes)')
print(f'POI type assortativity: {poi_assort:+.4f}')
print(f'  (positive = same-type POIs tend to co-occur)')

# POI type mixing matrix
type_pairs = []
for u, v in G.edges():
    tu = G.nodes[u].get('poi_type', 'Unknown')
    tv = G.nodes[v].get('poi_type', 'Unknown')
    type_pairs.append((tu, tv))
    type_pairs.append((tv, tu))

pair_df  = pd.DataFrame(type_pairs, columns=['Source', 'Target'])
mixing   = pair_df.groupby(['Source', 'Target']).size().unstack(fill_value=0)
mix_norm = mixing.div(mixing.sum(axis=1), axis=0)   # row-normalised

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    mix_norm, annot=True, fmt='.2f', cmap='Blues', linewidths=0.3,
    ax=ax, annot_kws={'size': 7},
    cbar_kws={'label': 'Row-normalised fraction of edges'}
)
ax.set_title('POI Type Mixing Matrix  (row-normalised)', fontweight='bold')
ax.set_xlabel('Target POI Type')
ax.set_ylabel('Source POI Type')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(rotation=0,  fontsize=8)
plt.tight_layout()
plt.show()